# 02 — Feature Engineering
## Phishing Email Detection with Explainable AI

Based on EDA findings, we extract **interpretable features** across three domains:
- **Textual features** — urgency language, keyword presence, text statistics
- **URL-based features** — link presence, IP-based URLs, suspicious patterns
- **Structural features** — email composition indicators

Additionally, a **Kazakhstan-oriented URL feature set** is integrated to support regional phishing detection.

---

In [ ]:
# ── Connect Google Drive ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

PHISHING_CSV    = '/content/drive/MyDrive/phishing_email.csv'
KZ_CSV          = '/content/drive/MyDrive/kazakhstan_urls.csv'
FEATURES_CSV    = '/content/drive/MyDrive/phishing_features.csv'
X_TEST_CSV      = '/content/drive/MyDrive/X_test.csv'
Y_TEST_CSV      = '/content/drive/MyDrive/y_test.csv'
XGB_MODEL_PATH  = '/content/drive/MyDrive/xgb_model.joblib'
print("Drive connected. Files ready.")

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv(PHISHING_CSV)
print(f'Loaded: {df.shape}')

## 1. Textual Features

In [ ]:
def extract_textual_features(text):
    text = str(text).lower()
    words = text.split()
    return {
        'char_count':        len(text),
        'word_count':        len(words),
        'avg_word_length':   np.mean([len(w) for w in words]) if words else 0,
        'unique_word_ratio': len(set(words)) / max(len(words), 1),
        'exclamation_count': text.count('!'),
        'question_count':    text.count('?'),
        'digit_ratio':       sum(c.isdigit() for c in text) / max(len(text), 1),
        'capital_ratio':     sum(c.isupper() for c in text) / max(len(text), 1),
        # Urgency language
        'has_urgent':        int(bool(re.search(r'\b(urgent|immediately|act now|verify now|limited time|expires)\b', text))),
        'has_free':          int(bool(re.search(r'\bfree\b', text))),
        'has_winner':        int(bool(re.search(r'\b(winner|won|congratulations|prize)\b', text))),
        'has_click_here':    int(bool(re.search(r'click here', text))),
        'has_password':      int(bool(re.search(r'\bpassword\b', text))),
        'has_account':       int(bool(re.search(r'\baccount\b', text))),
        'has_verify':        int(bool(re.search(r'\b(verify|confirm|validate)\b', text))),
        'has_suspend':       int(bool(re.search(r'\b(suspend|disabled|blocked|locked)\b', text))),
        'has_bank':          int(bool(re.search(r'\b(bank|credit|debit|transaction|wire)\b', text))),
        'has_offer':         int(bool(re.search(r'\b(offer|deal|discount|sale|promo)\b', text))),
        'has_personal_info': int(bool(re.search(r'\b(ssn|social security|date of birth|credit card)\b', text))),
    }

print('Extracting textual features...')
text_feats = df['text_combined'].apply(extract_textual_features)
text_df = pd.DataFrame(text_feats.tolist())
print(f'Textual features: {text_df.shape[1]}')
text_df.head(3)

## 2. URL-Based Features

In [ ]:
def extract_url_features(text):
    text = str(text).lower()
    urls = re.findall(r'http[s]?://[^\s]+', text)
    all_urls_str = ' '.join(urls)

    return {
        'url_count':           len(urls),
        'has_url':             int(len(urls) > 0),
        'has_https':           int('https://' in text),
        'has_http_only':       int('http://' in text and 'https://' not in text),
        'has_ip_url':          int(bool(re.search(r'http[s]?://\d{1,3}\.\d{1,3}\.\d{1,3}', text))),
        'has_shortened_url':   int(bool(re.search(r'(bit\.ly|tinyurl|goo\.gl|t\.co|ow\.ly|rb\.gy|is\.gd)', text))),
        'has_at_symbol_url':   int('@' in all_urls_str),
        'has_double_slash':    int('//' in all_urls_str.replace('://', '')),
        'has_hex_url':         int(bool(re.search(r'%[0-9a-f]{2}', all_urls_str))),
        'has_suspicious_tld':  int(bool(re.search(r'\.(xyz|top|click|gq|ml|cf|ga|tk|pw|cc)(/|$)', all_urls_str))),
        'max_url_length':      max((len(u) for u in urls), default=0),
        'has_subdomain_excess': int(any(u.count('.') > 4 for u in urls)),
    }

print('Extracting URL features...')
url_feats = df['text_combined'].apply(extract_url_features)
url_df = pd.DataFrame(url_feats.tolist())
print(f'URL features: {url_df.shape[1]}')
url_df.head(3)

## 3. Kazakhstan-Specific URL Feature

As stated in the theoretical part (Section 2.4), the system incorporates a regional dataset of Kazakhstani URLs to improve local phishing detection. Legitimate KZ URLs are sourced from official government portals (e.g., *egov.kz*), national banking institutions, and widely used local services. Suspicious KZ-themed URLs exhibit phishing characteristics such as domain impersonation or atypical TLDs.

In [ ]:
# --- Kazakhstan Legitimate Domains ---
kz_legit_domains = [
    'egov.kz', 'gov.kz', 'akorda.kz', 'minfin.kz', 'mfa.gov.kz',
    'enpf.kz', 'kaspi.kz', 'halykbank.kz', 'bcc.kz', 'atfbank.kz',
    'jusan.kz', 'kcell.kz', 'beeline.kz', 'activ.kz', 'kolesa.kz',
    'krisha.kz', 'olx.kz', 'satu.kz', 'nur.kz', 'informburo.kz'
]

# --- Kazakhstan Suspicious/Phishing URL Patterns ---
kz_phishing_patterns = [
    r'egov[\-\.][a-z0-9]+\.(?!kz)',         # egov impersonation on non-.kz
    r'kaspi[\-\.][a-z0-9]+\.(?!kz)',         # kaspi bank impersonation
    r'halyk[\-\.][a-z0-9]+\.(?!kz)',         # halyk bank impersonation
    r'gov-kz\.',                              # gov-kz lookalike
    r'\.kz\.[a-z]{2,4}/',                    # .kz embedded in longer domain
    r'login.*egov',                           # login page mimicking egov
    r'secure.*kaspi',                         # fake kaspi security page
    r'enpf.*bonus',                           # fake ENPF bonus claim
    r'gosuslugi.*kz',                         # Russian gov portal impersonation
]

def kz_url_feature(text):
    text = str(text).lower()
    urls = re.findall(r'http[s]?://[^\s]+', text)
    all_text_urls = ' '.join(urls) + ' ' + text

    # Check if any legitimate KZ domain is present
    has_kz_legit = int(any(d in text for d in kz_legit_domains))

    # Check for KZ phishing patterns
    has_kz_phishing = int(any(
        bool(re.search(p, all_text_urls)) for p in kz_phishing_patterns
    ))

    # .kz domain present at all
    has_kz_domain = int('.kz' in text)

    return {
        'has_kz_domain':    has_kz_domain,
        'has_kz_legit':     has_kz_legit,
        'has_kz_phishing':  has_kz_phishing,
        'kz_risk_score':    has_kz_phishing * 2 + has_kz_domain - has_kz_legit
    }

print('Extracting Kazakhstan URL features...')
kz_feats = df['text_combined'].apply(kz_url_feature)
kz_df = pd.DataFrame(kz_feats.tolist())
print(f'Kazakhstan features: {kz_df.shape[1]}')
print(f'Emails with .kz domain: {kz_df["has_kz_domain"].sum()}')
print(f'Emails with KZ phishing pattern: {kz_df["has_kz_phishing"].sum()}')

## 4. Combine All Features

In [ ]:
feature_df = pd.concat([text_df, url_df, kz_df], axis=1)
feature_df['label'] = df['label'].values
feature_df['text_combined'] = df['text_combined'].values

print(f'Total interpretable features: {feature_df.shape[1] - 2}')
print(f'Dataset size: {feature_df.shape[0]:,} rows')
feature_df.head(3)

## 5. Feature Correlation with Target

In [ ]:
from scipy.stats import pointbiserialr

feature_cols = [c for c in feature_df.columns if c not in ['label', 'text_combined']]
corr_target = {}
for col in feature_cols:
    try:
        r, _ = pointbiserialr(feature_df[col].fillna(0), feature_df['label'])
        corr_target[col] = r
    except:
        pass

corr_series = pd.Series(corr_target).sort_values()

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_series.values]
corr_series.plot.barh(ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Figure 3.6. Feature Correlation with Target (Point-Biserial)', fontsize=13, fontweight='bold')
ax.set_xlabel('Correlation with label (1=Phishing)')
plt.tight_layout()
plt.savefig('fig_3_6_feature_correlation.png', bbox_inches='tight')
plt.show()

print('\nTop 10 phishing indicators:')
print(corr_series.tail(10).round(3).to_string())
print('\nTop 10 legitimate indicators:')
print(corr_series.head(10).round(3).to_string())

## 6. Feature Summary Table

In [ ]:
feature_summary = pd.DataFrame({
    'Feature': feature_cols,
    'Domain': (['Textual'] * len(text_df.columns) +
               ['URL'] * len(url_df.columns) +
               ['KZ-Regional'] * len(kz_df.columns)),
    'Corr_with_Phishing': [corr_target.get(f, 0) for f in feature_cols],
    'Phishing_mean': [feature_df[feature_df['label']==1][f].mean() for f in feature_cols],
    'Legit_mean':    [feature_df[feature_df['label']==0][f].mean() for f in feature_cols],
}).sort_values('Corr_with_Phishing', ascending=False)

print('=== Feature Engineering Summary ===')
print(feature_summary.to_string(index=False))

# Save processed dataset
feature_df.to_csv(FEATURES_CSV, index=False)
print('\nSaved: phishing_features.csv')

## 7. Summary

| Feature Domain | Count | Key Indicators |
|---------------|-------|----------------|
| Textual | 17 | has_urgent, has_free, has_winner, has_verify |
| URL-based | 12 | has_url, has_ip_url, has_shortened_url, has_suspicious_tld |
| KZ-Regional | 4 | has_kz_phishing, kz_risk_score |
| **Total** | **33** | — |

All features are **human-interpretable**, directly supporting the XAI explanations generated in notebook 04.  
The dataset is saved as `phishing_features.csv` for use in modeling.